# 06 비교 보고서와 배포 전 판단

목표: 그룹 bootstrap·검증 규칙·실험 manifest와 실제 실행 HTML 보고서.

**준비:** 저장소 터미널에서 python -m pip install -r requirements/week11-offline.txt 실행.
VS Code 커널에서 저장소 .venv의 **Python 3.11** 선택 후 위에서 아래로 실행합니다.
각 파일은 다른 Notebook의 변수를 요구하지 않습니다. GPU·Ollama·API 키는 필요 없습니다.

**데이터:** 교수 작성 가상 쇼핑 정책입니다. 실제 기업 규정이나 고객 기록이 아닙니다.
핵심 단계는 수업에서 함께 실행하고 확장 실험은 복습용입니다. [실습 안내](../README.md).


In [ ]:
from pathlib import Path
import sys, json

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week11/lab/evaluation.py").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("저장소 안에서 Notebook을 실행하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
import pandas as pd
from IPython.display import display
from week11.lab.evaluation import load_data, run_experiment

docs, questions = load_data()
dev = [q for q in questions if q["split"] == "dev"]
test = [q for q in questions if q["split"] == "test"]
print("환경 준비 완료", sys.version.split()[0], "문서", len(docs), "질문", len(questions))

## 1. 전체 파이프라인

dev 선택 후 test 평가와 판단을 한 실행으로 묶습니다.

In [ ]:
result = run_experiment()
print("run_id:", result["run_id"])
print(result["manifest"]["candidate"])
display(pd.DataFrame(result["summary"]))

**확인과 해석**

ID는 데이터·코드·환경·설정에서 계산됩니다. 작은 표본의 한계는 남습니다.

## 2. 짝 비교

같은 의도의 두 표현을 함께 재표집합니다.

In [ ]:
ci = result["confidence"]
print(json.dumps(ci, indent=2))
print("차이 %p:", round(ci["delta"] * 100, 2))
print("95% 구간 %p:", [round(ci[k] * 100, 2) for k in ["low", "high"]])

**확인과 해석**

0 포함 시 이 절차에서 개선 방향이 명확하지 않습니다. 과업 밖 일반화 근거는 별도입니다.

## 3. 개별 변화

좋아진 사례와 나빠진 사례를 모두 남깁니다.

In [ ]:
a = {r["id"]: r for r in result["rows"]["baseline"]}
changes = [
    {
        "id": r["id"],
        "query": r["query"],
        "delta": r["recall"] - a[r["id"]]["recall"],
        "error": r["error"],
    }
    for r in result["rows"]["candidate"]
    if r["recall"] is not None
]
display(pd.DataFrame(changes).sort_values("delta"))

**확인과 해석**

평균 상승은 모든 질문의 개선을 뜻하지 않습니다.

## 4. 학습용 배포 기준

기권만 늘려 사용성을 잃는 후보도 걸러냅니다.

In [ ]:
print(result["gate"]["decision"])
display(pd.DataFrame(list(result["gate"]["checks"].items()), columns=["조건", "통과"]))

**확인과 해석**

HOLD는 실습 실패가 아닌 판단 결과입니다. REVIEW도 자동 배포가 아닙니다. 수치 기준은 교육용입니다.

## 5. HTML 보고서

실제 결과를 담아 브라우저에서 필터와 근거 탐색을 사용합니다.

In [ ]:
from week11.lab.run_eval import render_report

out = ROOT / "week11/lab/reports"
out.mkdir(exist_ok=True)
(out / "results.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
)
(out / "report.html").write_text(render_report(result), encoding="utf-8")
print("week11/lab/reports/report.html을 브라우저로 여세요.")

**확인과 해석**

데이터 내장 HTML이라 인터넷 없이 열립니다. reports는 Git에서 제외됩니다.

## 6. 재현 확인

같은 환경의 결정적 실험은 같은 결과를 내야 합니다.

In [ ]:
repeat = run_experiment()
assert repeat["run_id"] == result["run_id"]
assert repeat["summary"] == result["summary"]
print("재현 확인:", repeat["run_id"])
print(list(result["manifest"]))

**확인과 해석**

확률적 LLM은 반복 실행과 변동 폭을 추가 기록합니다.

## 확장 실험과 프로젝트 적용

1. 실패 두 개와 후속 실험 하나를 선택합니다.
2. python -m week11.lab.run_eval --enforce-gate는 HOLD일 때 종료 코드 2입니다. 의도한 차단입니다.
3. 이 실습으로 새 과제나 제출 마감이 생기지는 않습니다.

**실행 실패 시:** ModuleNotFoundError는 커널과 설치 터미널이 같은 .venv인지 확인합니다.
ROOT 오류는 저장소 안에서 파일을 열면 해결됩니다. 변수가 섞였다면 Restart Kernel 후 Run All.
예외 실험은 try/except로 설명하므로 정상 실행에서는 빨간 오류로 끝나지 않습니다.

**기록:** 변경 설정 / 예상 / 실제 관찰 / 실패 질문 ID / 다음 행동을 5줄로 적습니다.
개인 프로젝트 원문·연락처·API 키는 공개 GitHub에 올리지 않습니다.
